# Direct Multi-Step — Decision Tree baseline (10 model)

10 model độc lập, model thứ `h` học `features(t) → Weekly_Sales(t+h)`, h = 1..10.
Cùng feature, cùng tập train/val, cùng baseline Naive-52 với bản Random Forest để so trực tiếp.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

In [ ]:
def get_file_path(filename):
    current_dir = Path.cwd()
    for search_root in [current_dir] + list(current_dir.parents):
        for file in search_root.rglob(filename):
            if file.is_file():
                return file
    raise FileNotFoundError(f"Không tìm thấy file {filename}!")


train_df = pd.read_csv(get_file_path("train_final.csv"))
val_df = pd.read_csv(get_file_path("val_set.csv"))

for df in (train_df, val_df):
    df["Date"] = pd.to_datetime(df["Date"])

print("train :", train_df.shape, "|", train_df["Date"].min().date(), "->", train_df["Date"].max().date())
print("val   :", val_df.shape, "|", val_df["Date"].min().date(), "->", val_df["Date"].max().date())

In [ ]:
HORIZON = 10


def add_targets(df, horizon=HORIZON):
    d = df.sort_values(["Store", "Date"]).reset_index(drop=True).copy()
    for h in range(1, horizon + 1):
        d[f"target_t+{h}"] = d.groupby("Store")["Weekly_Sales"].shift(-h)
    return d


train_set = add_targets(train_df)
val_set = add_targets(val_df)

target_cols = [f"target_t+{h}" for h in range(1, HORIZON + 1)]
# visual trên store 1, bảng target
display(train_set.loc[train_set["Store"] == 1, ["Store", "Date", "Weekly_Sales"] + target_cols[:4]].head(8))

In [ ]:
drop_cols = ["Date", "Weekly_Sales", "Type"] + target_cols
feature_cols = [c for c in train_set.columns if c not in drop_cols]

print(f"{len(feature_cols)} cột feature:")
print(feature_cols)

obj_cols = [c for c in feature_cols if train_set[c].dtype == object]
assert not obj_cols, f"Còn cột chuỗi trong feature: {obj_cols}"

In [ ]:
# giữ đúng tham số tương ứng bản RF để so được: khác biệt chỉ nằm ở 1 cây vs 200 cây
dt_params = dict(max_depth=12, min_samples_split=5, min_samples_leaf=2, random_state=42)

dt_models, metrics, pred_frames = {}, [], []

for h in range(1, HORIZON + 1):
    target = f"target_t+{h}"
    train_c = train_set.dropna(subset=[target])
    val_c   = val_set.dropna(subset=[target])
    X_train, y_train = train_c[feature_cols], train_c[target]
    X_val,   y_val   = val_c[feature_cols],   val_c[target]

    dt = DecisionTreeRegressor(**dt_params).fit(X_train, y_train)
    preds = dt.predict(X_val)
    dt_models[h] = dt

    pred_frames.append(pd.DataFrame({
        "Store": val_c["Store"],
        "Date": val_c["Date"],
        "target_Date": val_c["Date"] + pd.Timedelta(days=h * 7),
        "horizon": h,
        "y_true": y_val,
        "y_pred": preds
    }))

    mae  = mean_absolute_error(y_val, preds)
    rmse = root_mean_squared_error(y_val, preds)
    wape = np.abs(y_val - preds).sum() / np.abs(y_val).sum()
    metrics.append({"horizon": f"t+{h:02d}", "n_train": len(train_c), "n_val": len(val_c),
                    "RMSE": rmse, "WAPE": wape, "MAE": mae})
    print(f"t+{h:02d} | RMSE {rmse:10,.0f} | WAPE {wape:6.2%} | MAE {mae:10,.0f}")

predictions = pd.concat(pred_frames, ignore_index=True)
metrics_df  = pd.DataFrame(metrics).set_index("horizon")

In [ ]:
view = metrics_df.copy()
view["WAPE"] = (view["WAPE"] * 100).round(2).astype(str) + "%"
display(view.round({"RMSE": 0, "MAE": 0}))

### So sánh công bằng giữa các horizon

Mỗi horizon chấm trên tập val khác nhau (h=1 có 30 tuần gốc, h=10 còn 21), nên lọc về chung 21 tuần gốc đầu. Chỉ lọc lúc báo cáo, không đụng lúc fit.

In [ ]:
all_dates = sorted(val_set["Date"].unique())
common_dates = all_dates[:-HORIZON]
common_df = predictions[predictions["Date"].isin(common_dates)].copy()

records = []
for h in range(1, HORIZON + 1):
    sub = common_df[common_df["horizon"] == h]
    y_true, y_pred = sub["y_true"], sub["y_pred"]
    records.append({
        "Horizon": f"t+{h:02d}",
        "Số dòng": len(sub),
        "RMSE": f"{root_mean_squared_error(y_true, y_pred):,.0f}",
        "WAPE": f"{np.abs(y_true - y_pred).sum() / np.abs(y_true).sum() * 100:.2f}%",
        "MAE": f"{mean_absolute_error(y_true, y_pred):,.0f}"
    })

print(f"Đánh giá trên {len(common_dates)} tuần gốc: "
      f"{pd.to_datetime(common_dates[0]).date()} -> {pd.to_datetime(common_dates[-1]).date()}")
print()
display(pd.DataFrame(records).set_index("Horizon"))

# baseline Naive-52

Dự báo tuần đích = doanh số thật của chính tuần đó năm ngoái, lấy từ `Lag_52` tại `target_Date`.

In [ ]:
common_df = common_df.merge(
    val_set[["Store", "Date", "Lag_52"]],
    left_on=["Store", "target_Date"],
    right_on=["Store", "Date"],
    suffixes=("", "_lookup")
)
assert common_df["Lag_52"].notna().all(), "thiếu Lag_52 cho một số tuần đích"

In [ ]:
MODEL_NAME = "DTree_direct"

cmp = []
for h in range(1, HORIZON + 1):
    s = common_df[common_df["horizon"] == h]
    w_dt  = np.abs(s["y_true"] - s["y_pred"]).sum() / np.abs(s["y_true"]).sum()
    w_n52 = np.abs(s["y_true"] - s["Lag_52"]).sum() / np.abs(s["y_true"]).sum()
    cmp.append({"model": MODEL_NAME, "horizon": h, "n": len(s),
                "WAPE": w_dt, "WAPE_naive52": w_n52, "skill": 1 - w_dt / w_n52})

cmp = pd.DataFrame(cmp)

view = cmp.set_index("horizon")[["n", "WAPE", "WAPE_naive52", "skill"]].copy()
for c in ("WAPE", "WAPE_naive52", "skill"):
    view[c] = (view[c] * 100).round(2).astype(str) + "%"
display(view)

print(f"{MODEL_NAME}: {cmp['WAPE'].mean():.2%} | Naive-52: {cmp['WAPE_naive52'].mean():.2%}"
      f" | skill trung bình {cmp['skill'].mean():+.1%}  (dương = thắng baseline)")

# để dành so với RF / recursive sau:
# cmp.to_csv(f"cmp_{MODEL_NAME}.csv", index=False)